In [0]:
from pyspark.sql import functions as F

df_silver = spark.read.table("workspace.jamal.silver_customers")

# ── Gold Table 1: Customers per state
customers_by_state = df_silver \
    .groupBy("customer_state") \
    .agg(
        F.count("customer_id").alias("total_customers"),
        F.countDistinct("customer_unique_id").alias("unique_customers"),
        F.countDistinct("customer_city").alias("distinct_cities")
    ) \
    .withColumn("repeat_customer_rate",
                F.round(
                    1 - (F.col("unique_customers") / F.col("total_customers")), 4
                )) \
    .orderBy(F.desc("total_customers"))

customers_by_state.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.jamal.gold_customers_by_state")

# ── Gold Table 2: Top cities per state
customers_by_city = df_silver \
    .groupBy("customer_state", "customer_city") \
    .agg(
        F.count("customer_id").alias("total_customers"),
        F.countDistinct("customer_unique_id").alias("unique_customers")
    ) \
    .orderBy(F.desc("total_customers"))

customers_by_city.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.jamal.gold_customers_by_city")

# ── Gold Table 3: ZIP code distribution (useful for map visuals in Power BI)
customers_by_zip = df_silver \
    .groupBy("customer_zip_code_prefix", "customer_state") \
    .agg(F.count("customer_id").alias("total_customers")) \
    .orderBy(F.desc("total_customers"))

customers_by_zip.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.jamal.gold_customers_by_zip")

# ── Verify all gold tables
display(spark.sql("SELECT * FROM workspace.jamal.gold_customers_by_state"))
display(spark.sql("SELECT * FROM workspace.jamal.gold_customers_by_city LIMIT 20"))